# Records: build once, reopen by key

A **record** is a directory that holds the result of one computation. Its name is the SHA-256
**key** of three things: the record type, the **inputs** it is computed from and the
**settings** of the computation. Calling a record type reopens the record if it exists and
builds it otherwise.

The build is the record type's `__init__`, and its signature is the interface:

- parameters before `/` (positional-only) are the inputs;
- parameters after `*` (keyword-only) are the settings, collected in a generated `Spec`.

This notebook defines record types, builds and reopens records, reads what they store, and
shows which changes give a new key.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
from jaxtyping import Float, Int

import tensordir as td

root = Path(tempfile.mkdtemp(prefix="tensordir-records-"))
td.set_root(root)  # every record in this notebook goes under this temporary directory

## The smallest record

`__init__` computes the result and assigns it to attributes, which are stored. The first call
runs `__init__`; later calls with the same settings reopen the stored record.

In [ ]:
class Squares(td.Record):
    def __init__(self, /, *, n: int):
        print("computing", n, "squares")
        self.values = np.arange(n) ** 2


print(Squares(n=4).values[:])  # .values is a lazy zarr array; [:] reads it
print(Squares(n=4).values[:])  # reopened: nothing printed by __init__

## A record type

A `Record` subclass can annotate the fields it stores. jaxtyping
annotations name the axes of array fields: a size `n_<things>` becomes the dimension `<thing>`,
so `"n_points n_dims"` is stored as `("point", "dim")`. The type id, part of every key, is the
class name; `__record_name__` overrides it, for example to keep keys across a rename.

In [ ]:
class Points(td.Record):
    xy: Float[np.ndarray, "n_points n_dims"]

    def __init__(self, /, *, n_points: int = 300, seed: int = 0):
        rng = np.random.default_rng(seed)
        centres = np.array([[-2.0, 0.0], [2.0, 0.0], [0.0, 3.0]])
        self.xy = centres[rng.integers(0, 3, n_points)] + rng.normal(scale=0.6, size=(n_points, 2))

Settings can be specs themselves. A `td.Spec` subclass is a frozen, keyword-only dataclass;
used as a setting, its fields enter the key with dotted names (`optimizer.lr`). Two field
options control the key:

- `td.field(default, hashed=False)` keeps a setting out of the key, for settings that do not
  change the result (device, number of workers);
- `td.field(default, since=v)` keeps a setting out of the key while it equals its default, so
  adding it later leaves existing keys unchanged.

In [ ]:
class Optimizer(td.Spec):
    lr: float = 0.1
    n_iterations: int = 20


builds = []  # counts how often the KMeans build runs in this process


class KMeans(td.Record):
    centres: Float[np.ndarray, "n_clusters n_dims"]
    labels: Int[np.ndarray, "n_points"]

    def __init__(
        self,
        points: Points,  # an input: enters the key by its type and key
        /,
        *,
        n_clusters: int,  # settings: the generated KMeans.Spec
        seed: int = 0,
        optimizer: Optimizer = Optimizer(),  # a nested spec
        device: str = td.field("cpu", hashed=False),
    ):
        builds.append(n_clusters)
        xy = points.xy
        rng = np.random.default_rng(seed)
        centres = xy[rng.choice(len(xy), n_clusters, replace=False)]
        for _ in range(optimizer.n_iterations):
            labels = np.argmin(((xy[:, None] - centres[None]) ** 2).sum(-1), axis=1)
            target = np.stack([xy[labels == k].mean(0) for k in range(n_clusters)])
            centres = centres + optimizer.lr * 10 * (target - centres)
        self.centres = centres
        self.labels = labels
        self["summary"] = {"inertia": float(((xy - centres[labels]) ** 2).sum())}  # stored as JSON


KMeans.Spec(n_clusters=3)  # the generated spec; the notebook repr marks settings outside the key

## Build and reopen

`KMeans(points, n_clusters=3)` returns a finished record: it runs `__init__` the first time and
reopens the record afterwards, without running `__init__` again. An input record enters the key
by its type and key, an array by a digest of its bytes, other values by their canonical JSON
form.

In [ ]:
points = Points()
fit = KMeans(points, n_clusters=3)  # builds
again = KMeans(Points(), n_clusters=3)  # reopens: same type, inputs and settings
print(len(builds), "build;", "same record:", again == fit)
print(fit.status, fit.key[:16], fit.path.relative_to(root))

The notebook repr is the record's tree: its type, key, status and stored fields.

In [ ]:
fit

## Read fields

- `record["name"]` returns a lazy handle: a `zarr.Array` for arrays, nothing is read yet.
- `record.name` returns the same lazy handle, except for fields typed as numpy arrays, which it
  reads into an ndarray.
- `record.load("name")` always reads into memory.
- `record.to_xarray("name")` labels an array with its stored dimension names.

In [ ]:
print(type(fit.centres).__name__, fit.centres.shape)
print(type(fit["centres"]).__name__, fit["centres"].shape)
print(fit.summary, fit.keys(), "labels" in fit)
fit.to_xarray("centres")

`record.meta` is the content of `_meta.json`: the full key, the stored spec (including
unhashed settings), the inputs by identity, status, timestamps, the field manifest and
provenance (host, user, job id, package versions, git commit of the repository defining the
class). `record.spec` revives the spec, `record.inputs` the inputs.

In [ ]:
print(sorted(fit.meta))
print(fit.meta["inputs"])
print(fit.spec)
print(fit.inputs[0] == points, sorted(fit.meta["provenance"]))

## Specs are values

`replace` returns a modified copy (nested names use `__` or dots), `flat` lists every setting
by dotted name, `diff` compares two specs, and `**spec` passes a spec's settings to a call.

In [ ]:
spec = fit.spec
other = spec.replace(seed=1, optimizer__lr=0.01, device="cuda")
print(other.flat())
print(spec.diff(other))
print("key-relevant only:", spec.diff(other, hashed=True))
print(KMeans(points, **spec) == fit)

## Which changes give a new key

The key covers the record type, `schema_version`, the hashed settings and the inputs. The
unhashed `device` does not change it; another `Points` input does. A float setting given an int
is stored as a float, so `lr=1` and `lr=1.0` share a key.

In [ ]:
gpu = KMeans(points, n_clusters=3, device="cuda")
print("device changes the key:", gpu.key != fit.key, "| builds so far:", len(builds))

refit = KMeans(Points(seed=1), n_clusters=3)
print("another input changes the key:", refit.key != fit.key, "| builds so far:", len(builds))
print(KMeans.compute_key(points, n_clusters=3, optimizer=Optimizer(lr=1)) == KMeans.compute_key(
    points, n_clusters=3, optimizer=Optimizer(lr=1.0)
))

Inputs enter the key by position, not by name: renaming the `points` parameter keeps every
key. Arrays passed as inputs are hashed by digest; `td.Digest.of(array)` gives the same
fingerprint for use in a setting.

In [ ]:
class Weighted(td.Record):
    def __init__(self, points: Points, weights: np.ndarray, /, *, n_clusters: int = 3):
        self.total = float(weights.sum())


weights = np.linspace(0.5, 1.5, 300)
k1 = Weighted.compute_key(points, weights)  # no filesystem access
k2 = Weighted.compute_key(points, weights.copy())
k3 = Weighted.compute_key(points, weights + 1e-9)
print(k1 == k2, k1 == k3)
td.Digest.of(weights)

## Look without building

- `locate(...)` returns the record handle whatever its status; nothing is built.
- `lookup(...)` returns the finished record or `None`.
- `compute_key(...)` returns the full key without touching the filesystem.

Each takes the same arguments as the call.

In [ ]:
missing = KMeans.locate(points, n_clusters=5)
print(missing.status, KMeans.lookup(points, n_clusters=5))
print(KMeans.lookup(points, n_clusters=3) == fit)

## Finished records are read-only

A key promises its contents, so a finished record refuses writes. Change the settings instead,
or use a workspace record (`mutable = True`, see the notebook on workspaces and stages).

In [ ]:
try:
    fit.centres = np.zeros((3, 2))
except td.ReadOnlyError as exc:
    print("ReadOnlyError:", exc)

## Adding a setting without orphaning records

Suppose `KMeans` later gains a `tolerance` setting. Declared with `since=`, it stays out of the
key while it equals its default, so existing records keep their keys. Queries and tables show
the default for records built before the setting existed. Redefining a class with the same name
in the same module replaces it, as in a notebook.

In [ ]:
class KMeans(td.Record):
    centres: Float[np.ndarray, "n_clusters n_dims"]

    def __init__(
        self,
        points: Points,
        /,
        *,
        n_clusters: int,
        seed: int = 0,
        optimizer: Optimizer = Optimizer(),
        device: str = td.field("cpu", hashed=False),
        tolerance: float = td.field(0.0, since=2),
    ):
        self.centres = points.xy[:n_clusters]


print(KMeans.compute_key(points, n_clusters=3) == fit.key)
print(KMeans.compute_key(points, n_clusters=3, tolerance=1e-4) == fit.key)
print(KMeans.lookup(points, n_clusters=3).spec)

A change to the build that invalidates old results calls for a new `schema_version`: it enters
every key of the type, so old records are no longer found.

In [ ]:
class KMeansV1(KMeans):
    __record_name__ = "KMeans"
    schema_version = 1


print(KMeansV1.lookup(points, n_clusters=3))

## Hand-named directories, deletion

`Points.at(path, ...)` builds or opens a record at a path you choose, as datamate's
`Directory(path, config)` did. `td.Record.at(path)` opens any record and returns it as its
registered class.

In [ ]:
named = Points.at(root / "runs" / "reference", n_points=50)
reopened = td.Record.at(root / "runs" / "reference")
print(type(reopened).__name__, reopened.spec, reopened.xy.shape)

`record.delete()` removes the directory; the next call builds it again.

In [ ]:
small = Points(n_points=10)
small.delete()
print(small.status, Points(n_points=10).status)

## On disk

Records live under `<root>/<type>/<first 16 hex digits of the key>/`; `.tensordir/` holds
leases, temporary builds and the index.

In [ ]:
td.Directory(root).tree(depth=3)